## Architecture Overview
```
User Query
    │
    ▼
LLM Filter Extractor      ──► Structured filters (fees, level, city, course_name, etc.)
    │
    ▼
Hybrid Search (pgvector)  ──► Top-N candidates by vector similarity + SQL filters
    │
    ▼
Deduplication             ──► Remove duplicate (college, course) pairs
    │
    ▼
Cross-Encoder Reranker    ──► Semantic relevance score for each candidate
    │
    ▼
Score Normalization       ──► Bring similarity, cross, ranking scores to [0,1]
    │
    ▼
RRF Fusion                ──► Combine all signals using Reciprocal Rank Fusion
    │
    ▼
Diversification           ──► One result per college (avoid repetition)
    │
    ▼
LLM Answer Generator      ──► Natural language answer from top results
```

## Step 1 — Install Dependencies

In [ ]:
!pip -q install psycopg2-binary python-dotenv pgvector sentence-transformers groq torch

## Step 2 — Imports

In [ ]:
import os
import re
import json
import psycopg2
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer, CrossEncoder
from groq import Groq

## Step 3 — Configuration

Set secrets as environment variables before running in production.

--- Model config ---

In [ ]:
# The bi-encoder used to turn text into vectors for pgvector search.
# Must match the model used when the embeddings were originally created and stored.
EMBEDDER_MODEL = os.getenv('EMBEDDER_TRANSFORMER_MODEL', 'all-MiniLM-L6-v2')

--- LLM model config ---

In [ ]:
# The Groq-hosted LLM used for filter extraction and final answer generation.
LLM_MODEL = os.getenv('LLM_MODEL', 'llama-3.3-70b-versatile')

--- Credentials ---

In [ ]:
# --- Model config ---
# The bi-encoder used to turn text into vectors for pgvector search.
# Must match the model used when the embeddings were originally created and stored.
EMBEDDER_MODEL = os.getenv('EMBEDDER_TRANSFORMER_MODEL', 'all-MiniLM-L6-v2')

# The Groq-hosted LLM used for filter extraction and final answer generation.
LLM_MODEL = os.getenv('LLM_MODEL', 'llama-3.3-70b-versatile')

# --- Credentials ---
POSTGRES_URL = os.getenv('POSTGRES_DATABASE_URL', None)
POSTGRES_PORT = os.getenv('POSTGRES_PORT', None)
POSTGRES_HOST = os.getenv('POSTGRES_HOST', None)
POSTGRES_DB_NAME = os.getenv('POSTGRES_DB_NAME', None)
POSTGRES_USER = os.getenv('POSTGRES_USER', None)
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD', None)
GROQ_API_KEY = os.getenv('GROQ_API_KEY', None)

print('Embedder model :', EMBEDDER_MODEL)
print('LLM model      :', LLM_MODEL)
print('Postgres URL set:', bool(POSTGRES_URL))
print('Groq key set    :', bool(GROQ_API_KEY))

## Step 4 — Database Connection

We connect directly with psycopg2 and register pgvector so that Python can
read/write `vector` columns natively.

In [ ]:
conn = psycopg2.connect(
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    dbname=POSTGRES_DB_NAME,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    sslmode='require'        # Aiven always requires SSL
)
register_vector(conn)        # Enables <=> cosine-distance operator in psycopg2
cursor = conn.cursor()
print('Database connection established.')

## Step 5 — Load ML Models

- **Bi-encoder** (`SentenceTransformer`): converts a query/doc to a dense vector.
  Used for the initial ANN (approximate nearest-neighbour) retrieval in pgvector.
- **Cross-encoder** (`CrossEncoder`): scores a (query, document) *pair* together.
  Much more accurate than bi-encoder but slower — used only on the shortlist.

In [ ]:
# Bi-encoder — same model that was used to generate the stored embeddings.
# Changing this model would make the query vector incompatible with the stored ones.
bi_encoder = SentenceTransformer(EMBEDDER_MODEL)

# Cross-encoder for reranking — ms-marco is trained on passage-retrieval so it
# works well for (query, college description) relevance scoring.
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

print('Models loaded successfully.')

## Step 6 — Hybrid Search with Filters

Combines:
- **Semantic search** via pgvector cosine distance (`<=>`) on the `embedding` column.
- **Hard filters** (fees, city, level, duration, course_name, etc.) applied as SQL WHERE clauses.

In [ ]:
def hybrid_search_with_filters(user_query: str, filters: dict = None, limit: int = 50) -> list[dict]:
    """
    Retrieve candidate college-courses using pgvector ANN search + SQL hard filters.

    Args:
        user_query: Raw natural-language query string.
        filters:    Dict of hard filters extracted by the LLM (see extract_filters_llm).
        limit:      Max rows to retrieve from the DB before reranking.

    Returns:
        List of dicts, one per (college, course) row, ordered by cosine similarity.
    """
    if filters is None:
        filters = {}

    try:
        # Embed the query with the same model used when ingesting data.
        query_vector = bi_encoder.encode(user_query).tolist()

        # -------------------------------------------------------------------
        # Base SELECT — the similarity_score alias is computed here so it can
        # be inspected later (for debugging / score normalisation).
        # %s placeholders are filled in order via params list.
        # -------------------------------------------------------------------
        sql = """
            SELECT
                college_name,
                course_name,
                city,
                fees,
                ranking,
                ranking_agency,
                eligibility,
                duration,
                course_type,
                level,
                job_roles,
                cutoff_score,
                admission_start_date,
                admission_end_date,
                semantic_text,
                1 - (embedding <=> %s::vector) AS similarity_score
            FROM college_courses
        """
        params = [query_vector]   # First param: embedding for the SELECT column
        conditions = []

        # -------------------------------------------------------------------
        # Hard filters — each one narrows the candidate set before ANN sort.
        # Keep filters loose (ILIKE) for text fields to handle case/spacing.
        # -------------------------------------------------------------------

        if filters.get('min_fees') is not None:
            conditions.append('fees >= %s')
            params.append(filters['min_fees'])

        if filters.get('max_fees') is not None:
            conditions.append('fees <= %s')
            params.append(filters['max_fees'])

        if filters.get('level'):
            # Using exact match now so 'Graduation' != 'Post Graduation'.
            conditions.append('LOWER(level) = LOWER(%s)')
            params.append(filters['level'])

        if filters.get('city'):
            conditions.append('city ILIKE %s')
            params.append(f"%{filters['city']}%")

        if filters.get('admission_open'):
            # Only return colleges whose admission window is still open.
            conditions.append('admission_end_date >= CURRENT_DATE')

        if filters.get('max_ranking') is not None:
            conditions.append('ranking <= %s')
            params.append(filters['max_ranking'])

        if filters.get('duration'):
            conditions.append('duration ILIKE %s')
            params.append(f"%{filters['duration']}%")

        if filters.get('course_name'):
            conditions.append('course_name ILIKE %s')
            params.append(f"%{filters['course_name']}%")

        if conditions:
            sql += ' WHERE ' + ' AND '.join(conditions)

        # ORDER BY vector distance — second embedding param for ORDER BY clause.
        sql += ' ORDER BY embedding <=> %s::vector LIMIT %s'
        params.append(query_vector)
        params.append(limit)

        cursor.execute(sql, tuple(params))
        rows = cursor.fetchall()

        columns = [
            'college_name', 'course_name', 'city', 'fees', 'ranking',
            'ranking_agency', 'eligibility', 'duration', 'course_type',
            'level', 'job_roles', 'cutoff_score', 'admission_start_date',
            'admission_end_date', 'semantic_text', 'similarity_score'
        ]
        return [dict(zip(columns, row)) for row in rows]

    except Exception as e:
        conn.rollback()   # Release the failed transaction so the connection stays usable.
        print(f'[hybrid_search] DB error: {e}')
        return []

## Step 7 — LLM Filter Extractor

Converts a free-text user query into a structured JSON filter dict.  
The LLM acts as a zero-shot slot-filler — no training required.

In [ ]:
# System prompt tells the LLM exactly which slots to fill and how.
# Keep it short and precise — verbose prompts increase hallucination risk.
FILTER_EXTRACTION_PROMPT = """
You are a structured data extractor. Your only job is to parse a college-search
query and return a JSON object with filter values.

Available filter keys (omit a key if the user did not mention it):
  max_fees        : integer (INR)
  min_fees        : integer (INR)
  level           : "Graduation" | "Post Graduation"
  admission_open  : true | false
  max_ranking     : integer
  city            : string
  duration        : string (e.g. "2 years", "3 years")
  course_name     : string — domain keyword like "law", "MBA", "engineering"

Rules:
- Fix obvious spelling mistakes in the query before parsing.
- Return ONLY valid JSON, no explanation.
- If no filter is found, return {}.
"""

llm_client = Groq(api_key=GROQ_API_KEY)

def extract_filters_llm(user_query: str) -> dict:
    """
    Use an LLM to extract structured hard filters from a natural-language query.

    Example:
        Input : 'Best law colleges under 1 lakh in Bangalore'
        Output: {'course_name': 'law', 'max_fees': 100000, 'city': 'Bangalore'}
    """
    prompt = f"{FILTER_EXTRACTION_PROMPT}\n\nUser Query: {user_query}\n\nJSON:"

    try:
        response = llm_client.chat.completions.create(
            model=LLM_MODEL,
            temperature=0, # Deterministic output for parsing
            messages=[{'role': 'user', 'content': prompt}]
        )
        raw_text = response.choices[0].message.content.strip()

        # Extract the first {...} block in case the LLM adds explanation text.
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if not match:
            print('[extract_filters_llm] No JSON found in LLM response.')
            return {}

        return json.loads(match.group(0))

    except Exception as e:
        print(f'[extract_filters_llm] Error: {e}')
        return {}

## Step 8 — Deduplication & Diversification

In [ ]:
def deduplicate(docs: list[dict]) -> list[dict]:
    """
    Remove exact (college_name, course_name) duplicates that can arise when the
    same row matches multiple filter branches.
    """
    seen = set()
    unique = []
    for doc in docs:
        key = (doc['college_name'], doc['course_name'])
        if key not in seen:
            seen.add(key)
            unique.append(doc)
    return unique

The diversify() function ensures that the final search results contain different colleges instead of many courses from the same college.

In [ ]:
def diversify(docs: list[dict], top_n: int = 10) -> list[dict]:
    """
    Return at most one result per college so the answer covers a wider range of
    institutions rather than showing 5 courses from the same college.

    Stops after `top_n` unique colleges.
    """
    seen_colleges = set()
    final = []
    for doc in docs:
        if doc['college_name'] not in seen_colleges:
            final.append(doc)
            seen_colleges.add(doc['college_name'])
        if len(final) == top_n:
            break
    return final

## Step 9 — Cross-Encoder Reranker

The bi-encoder embedding is fast but imprecise — it can't look at the query and
document *together*. The cross-encoder reads both at once and produces a much
more accurate relevance score.

We run it only on the shortlist (≤50 docs) to keep latency acceptable.

In [ ]:
def rerank(query: str, documents: list[dict]) -> list[dict]:
    """
    Score every (query, semantic_text) pair with the cross-encoder.
    Adds a `cross_score` field to each document dict.
    Returns documents sorted by cross_score descending.
    """
    if not documents:
        return documents

    # The cross-encoder expects a list of (query, passage) string pairs.
    pairs = [(query, doc['semantic_text'] or '') for doc in documents]
    scores = cross_encoder.predict(pairs)          # Returns a numpy array

    for doc, score in zip(documents, scores):
        doc['cross_score'] = float(score)

    # sort descending order
    return sorted(documents, key=lambda x: x['cross_score'], reverse=True)

## Step 10 — Score Helpers

--- Ranking score computation

College ranking is a positive integer (lower = better).  
We convert it to a quality score using `1 / ranking` — rank 1 → 1.0, rank 100 → 0.01.  
Then it's normalised with the rest.

In [ ]:
def add_ranking_score(documents: list[dict]) -> list[dict]:
    """
    Convert the raw `ranking` integer into a `ranking_score` float in (0, 1].
    ranking=1 → highest quality; None/0 → score 0.

    Must be called BEFORE normalize_scores so the values get scaled consistently.
    """
    for doc in documents:
        r = doc.get('ranking')
        doc['ranking_score'] = (1.0 / r) if r and r > 0 else 0.0
    return documents

--- Min-max normalisation

Brings any score column to [0, 1] so they can be combined fairly.

In [ ]:
def normalize_scores(documents: list[dict], key: str) -> list[dict]:
    """
    Min-max normalise a score field across all documents in-place.
    If all values are equal (max == min) every score is set to 0 to avoid /0.
    """
    values = [doc[key] for doc in documents if doc.get(key) is not None]
    if not values:
        return documents

    lo, hi = min(values), max(values)
    span = hi - lo

    for doc in documents:
        v = doc.get(key)
        doc[key] = ((v - lo) / span) if (v is not None and span > 0) else 0.0

    return documents

In [ ]:
def compute_final_score(doc: dict) -> float:
    """
    Weighted combination of three normalised signals:
      - similarity_score : bi-encoder cosine similarity (speed)
      - cross_score      : cross-encoder relevance      (accuracy)
      - ranking_score    : college ranking quality      (authority)

    Weights should sum to 1.0.
    """
    return (
        0.35 * doc['similarity_score'] +
        0.45 * doc['cross_score'] +
        0.20 * doc.get('ranking_score', 0.0)   # ranking_score now always set
    )

## Step 11 — Reciprocal Rank Fusion (RRF)

RRF is a robust rank-aggregation method that doesn't require calibrated scores —
only the rank positions matter.  
Formula for a document `d` across `n` ranked lists: `RRF(d) = Σ 1/(k + rank_i(d))`  
k=60 is the standard smoothing constant.

In [ ]:
def apply_rrf(docs: list[dict], k: int = 60) -> list[dict]:
    """
    Assign an `rrf_score` to each document by fusing three ranking signals:
      1. similarity_score rank
      2. cross_score rank
      3. ranking_score rank

    Higher rrf_score = more relevant.
    """
    # Assign rank positions for each signal (1 = best)
    for rank, doc in enumerate(
        sorted(docs, key=lambda x: x['similarity_score'], reverse=True)
    ):
        doc['sim_rank'] = rank + 1

    for rank, doc in enumerate(
        sorted(docs, key=lambda x: x['cross_score'], reverse=True)
    ):
        doc['cross_rank'] = rank + 1

    for rank, doc in enumerate(
        sorted(docs, key=lambda x: x.get('ranking_score', 0), reverse=True)
    ):
        doc['ranking_rank'] = rank + 1

    for doc in docs:
        doc['rrf_score'] = (
            1 / (k + doc['sim_rank']) +
            1 / (k + doc['cross_rank']) +
            1 / (k + doc['ranking_rank'])
        )

    return docs

## Step 12 — Context Builder & LLM Answer Generator

In [ ]:
def build_context(top_docs: list[dict]) -> str:
    """
    Serialise the top results into a plain-text block for the LLM prompt.
    Include every field that could be useful when answering the user's question.
    """
    lines = []
    for i, doc in enumerate(top_docs, 1):
        lines.append(f"--- Result {i} ---")
        lines.append(f"College      : {doc['college_name']}")
        lines.append(f"Course       : {doc['course_name']}")
        lines.append(f"Course Type  : {doc.get('course_type', 'N/A')}")
        lines.append(f"Level        : {doc.get('level', 'N/A')}")
        lines.append(f"City         : {doc['city']}")
        lines.append(f"Fees (INR)   : {doc['fees']}")
        lines.append(f"Duration     : {doc.get('duration', 'N/A')}")
        lines.append(f"Eligibility  : {doc.get('eligibility', 'N/A')}")
        lines.append(f"Ranking      : {doc.get('ranking', 'N/A')} ({doc.get('ranking_agency', 'N/A')})")   # Added
        lines.append(f"Cutoff Score : {doc.get('cutoff_score', 'N/A')}")
        lines.append(f"Job Roles    : {doc.get('job_roles', 'N/A')}")
        lines.append(f"Admission End: {doc.get('admission_end_date', 'N/A')}")
        lines.append('')
    return '\n'.join(lines)

In [ ]:
def generate_answer(query: str, context: str) -> str:
    """
    Use the LLM to produce a natural-language answer grounded in the retrieved context.
    The prompt instructs the model to stay within the provided data (no hallucination).
    """
    prompt = f"""\
You are an expert educational counsellor helping a student choose the right college.

Use ONLY the information in the context below to answer the question.
Do not mention colleges or courses that are not in the context.
If the context does not contain enough information, say so.

Context:
{context}

Student Question:
{query}

Answer:"""

    try:
        response = llm_client.chat.completions.create(
            model=LLM_MODEL,
            temperature=0.2,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f'[generate_answer] Error: {e}')
        return 'An error occurred while generating the answer.'

## Step 13 — Full Pipeline

Orchestrates every component in order.  
Returns both the structured top results and an LLM-generated answer.

In [ ]:
def full_pipeline(user_query: str) -> dict:
    """
    End-to-end RAG pipeline for college search.

    Args:
        user_query: The student's free-text question.

    Returns:
        {
          'answer'  : str   — LLM-generated natural-language answer,
          'results' : list  — top-10 ranked college-course dicts,
          'filters' : dict  — filters extracted from the query (for debugging),
        }
    """
    print(f'Query: {user_query}\n')

    # ------------------------------------------------------------------
    # 1. Extract structured filters from the query via LLM.
    #    FIX: was hardcoded in v1; now auto-extracted for every query.
    # ------------------------------------------------------------------
    filters = extract_filters_llm(user_query)
    print(f'Extracted filters: {filters}')

    # ------------------------------------------------------------------
    # 2. Hybrid search with all filters applied.
    # ------------------------------------------------------------------
    candidates = hybrid_search_with_filters(user_query, filters, limit=100)

    # Fallback: level data is often missing in DBs — retry without it.
    if not candidates and filters.get('level'):
        print('[pipeline] No results with level filter — retrying without it.')
        filters_without_level = {k: v for k, v in filters.items() if k != 'level'}
        candidates = hybrid_search_with_filters(user_query, filters_without_level, limit=100)

    # Fallback: duration data is often missing in DBs — retry without it.
    if not candidates and filters.get('duration'):
        print('[pipeline] No results with duration filter — retrying without it.')
        filters_without_duration = {k: v for k, v in filters.items() if k != 'duration'}
        candidates = hybrid_search_with_filters(user_query, filters_without_duration, limit=100)

    if not candidates:
        print('[pipeline] No candidates found.')
        return {'answer': 'No matching colleges found.', 'results': [], 'filters': filters}

    print(f'Candidates retrieved: {len(candidates)}')

    # ------------------------------------------------------------------
    # 3. Remove exact duplicates.
    # ------------------------------------------------------------------
    candidates = deduplicate(candidates)

    # Pre-filter to the top-40 by bi-encoder score before the expensive
    # cross-encoder pass (saves ~60% of cross-encoder inference time).
    candidates = sorted(candidates, key=lambda x: x['similarity_score'], reverse=True)[:40]

    # ------------------------------------------------------------------
    # 4. Cross-encoder reranking.
    # ------------------------------------------------------------------
    reranked = rerank(user_query, candidates)

    # ------------------------------------------------------------------
    # 5. Compute ranking_score from raw `ranking` integer.
    #    FIX: was never done in v1 so ranking_score was always 0.
    # ------------------------------------------------------------------
    add_ranking_score(reranked)

    # ------------------------------------------------------------------
    # 6. Normalise all three score columns to [0, 1].
    # ------------------------------------------------------------------
    normalize_scores(reranked, 'similarity_score')
    normalize_scores(reranked, 'cross_score')
    normalize_scores(reranked, 'ranking_score')

    # ------------------------------------------------------------------
    # 7. RRF fusion across all three signals.
    #    FIX: v1 only fused similarity + cross; ranking signal now included.
    # ------------------------------------------------------------------
    reranked = apply_rrf(reranked)
    reranked.sort(key=lambda x: x['rrf_score'], reverse=True)

    # ------------------------------------------------------------------
    # 8. Diversify: one result per college.
    # ------------------------------------------------------------------
    top_docs = diversify(reranked, top_n=10)

    # ------------------------------------------------------------------
    # 9. Generate LLM answer.
    #    FIX: was commented out in v1 — now active.
    # ------------------------------------------------------------------
    context = build_context(top_docs)
    answer = generate_answer(user_query, context)

    return {
        'answer' : answer,
        'results': top_docs,
        'filters': filters
    }

## Step 14 — Test the Pipeline

Run a sample query and inspect both the structured results and the LLM answer.

--- Primary test query ---

In [ ]:
query = 'Best law colleges with eligibility criteria LLB with 50%, good ranking with duration only 2 years in Bangalore'

result = full_pipeline(query)

print('\n' + '='*60)
print('LLM ANSWER')
print('='*60)
print(result['answer'])

print('\n' + '='*60)
print('TOP RESULTS')
print('='*60)
for i, doc in enumerate(result['results'], 1):
    print(f"{i}. {doc['college_name']} | {doc['course_name']} | {doc['city']} | Fees: {doc['fees']} | Rank: {doc.get('ranking')}")

--- Additional test queries to validate other filters ---

In [ ]:
queries = [
    'MBA colleges in Mumbai under 3 lakh fees',
    'Top engineering colleges with admission open in Chennai',
    'Post graduation medical colleges ranked under 50 in Delhi'
]

for q in queries:
    print(f'\nQuery: {q}')
    out = full_pipeline(q)
    print(f'Filters extracted: {out["filters"]}')
    print(f'Results found    : {len(out["results"])}')
    print('-' * 50)